#### 7.1. Verifying  libraries

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    try:
        v = version(lib)
        print(f'{lib}: {v}')
    except Exception as e:
        print(f'{lib}: Not found')

reasoning_from_scratch: Not found
torch: 2.4.1+cu124
tokenizers: 0.23.3


#### 7.2. Loading pretrained model

In [11]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using NVIDIA CUDA GPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


d:\Stuffs\FunProjects\DumLRM\src\dumlrm\chapters\reasoning_from_scratch\ch03.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path

In [12]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 To solve the equation "Half the value of \(3x - 9\) is \(x + 37\)," follow these steps:

1. **Set up the equation based on the given information:**
   \[
   \frac{1}{2} \times (3x - 9) = x + 37
   \]

2. **Eliminate the fraction by multiplying both sides by 2:**
   \[
   2 \times \frac{1}{2} \times (3x - 9) = 2 \times (x + 37)
   \]
   \[
   3x - 9 = 2x + 74
   \]

3. **Isolate the variable \(x\) by subtracting \(2x\) from both sides:**
   \[
   3x - 2x - 9 = 2x - 2x + 74
   \]
   \[
   x - 9 = 74
   \]

4. **Solve for \(x\) by adding 9 to both sides:**
   \[
   x - 9 + 9 = 74 + 9
   \]
   \[
   x = 83
   \]

**Final Answer:**
\[
\boxed{83}
\]

#### 7.3. Tracking GRPO performance metrics

##### 7.3.1 Advantages tracking

In [1]:
import torch

def compute_advantage_stats(rewards_list):
    rewards = torch.tensor(rewards_list)

    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()

    return advantages, adv_avg, adv_std

##### 7.3.2 Entropy tracking

In [4]:
# entropy = -1 * prob * log_prob

# Toy logits
logits = torch.tensor([
    0.6667, -2.0000,  1.3333, -0.0000, -0.6667,  2.0000, -1.3333
])

logprobs = torch.log_softmax(logits, dim=-1)
print("all token logprobs       : ", logprobs)

selected_token = torch.argmax(logprobs)
selected = logprobs[selected_token]
print("selected token           : ", selected_token)
print("selected token logprob   : ", selected)

all token logprobs       :  tensor([-2.0442, -4.7109, -1.3776, -2.7109, -3.3776, -0.7109, -4.0442])
selected token           :  tensor(5)
selected token logprob   :  tensor(-0.7109)


In [7]:
logprobs = torch.log_softmax(logits, dim=-1)
# optimization skipping probs
probs = torch.exp(logprobs)

# entropy
entropy = torch.sum(-1 * probs * logprobs)

print("probs    :", probs)
print("logprobs :", logprobs)
print("entropy  :", entropy)


probs    : tensor([0.1295, 0.0090, 0.2522, 0.0665, 0.0341, 0.4912, 0.0175])
logprobs : tensor([-2.0442, -4.7109, -1.3776, -2.7109, -3.3776, -0.7109, -4.0442])
entropy  : tensor(1.3700)


In [ ]:
# max_entropy = log(vocab_size)
vocab_size = 151_900
torch.log(torch.tensor(vocab_size))

tensor(11.9310)

In [13]:
def sequence_logprob_and_entropy(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)

    sequence_answer_logprob = selected[prompt_len - 1]

    logp_all_steps = torch.sum(sequence_answer_logprob)

    # calculate entropy
    all_answer_logprobs = logprobs[:-1][prompt_len - 1]
    if all_answer_logprobs.numel() == 0:
        entropy_all_steps = logp_all_steps.new_tensor(0.0)
    else:
        all_answer_probs = torch.exp(all_answer_logprobs)
        plogp = all_answer_probs * all_answer_logprobs
        step_entropy = -1 * torch.sum(plogp, dim=-1)
        entropy_all_steps = torch.mean(step_entropy)

    return logp_all_steps, entropy_all_steps